# 检查检索结果后再继续

Corrective RAG（CRAG）先检查检索结果的相关性，再决定是否用固定补查问题重查。Self-RAG 启发式流程则在生成草稿后严格反思是否覆盖问题并决定是否补查。两种流程的契约不满足时都会直接阻断，不能用默认值、原问题拼接或外部知识掩盖失败。

CRAG 重排器只从明确的本地 BAAI/bge-reranker-base snapshot 加载，并显式使用 local_files_only=True。运行 Notebook 不联网；首次运行前可按本节 README 的资源准备命令联网下载本地 snapshot。

In [1]:
import sys
from pathlib import Path

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / 'data' / 'dataset/manifest.json').is_file():
            return folder
    raise FileNotFoundError('没有找到教程数据目录，请从本节所在目录运行')

course_root = find_course_root(Path.cwd().resolve())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from common.eval_utils import build_bm25_search, load_query_catalog, load_pdf_pages
from common.nontraining_utils import load_annotation

data = load_query_catalog()
cases = {item['id']: item for item in data}
search = build_bm25_search(load_pdf_pages())

RERANKER_MODEL_ID = 'BAAI/bge-reranker-base'
from modelscope import snapshot_download
try:
    RERANKER_SNAPSHOT = Path(snapshot_download(RERANKER_MODEL_ID, local_files_only=True))
except Exception as error:
    raise FileNotFoundError(f'缺少明确的本地 {RERANKER_MODEL_ID} snapshot') from error
required_files = ('config.json', 'tokenizer_config.json')
missing_files = [name for name in required_files if not (RERANKER_SNAPSHOT / name).is_file()]
if not RERANKER_SNAPSHOT.is_dir() or missing_files or not any((RERANKER_SNAPSHOT / name).is_file() for name in ('model.safetensors', 'pytorch_model.bin')):
    missing = ', '.join(missing_files or ['model weights'])
    raise FileNotFoundError(f'缺少明确的本地 {RERANKER_MODEL_ID} snapshot 或文件：{RERANKER_SNAPSHOT}（{missing}）')
tokenizer = AutoTokenizer.from_pretrained(str(RERANKER_SNAPSHOT), local_files_only=True)
reviewer = AutoModelForSequenceClassification.from_pretrained(str(RERANKER_SNAPSHOT), local_files_only=True).eval()

def relevance_scores(question, results):
    batch = tokenizer([[question, item.text] for item in results], padding=True, truncation=True, max_length=512, return_tensors='pt')
    with torch.no_grad():
        return reviewer(**batch).logits.view(-1).tolist()

def pages_with_scores(results, scores):
    return [(item.page, round(score, 3)) for item, score in zip(results, scores)]

threshold = 1.0
main = cases['model_selection_no_absolute_best']
before = search(main['query'], top_k=4)
before_scores = relevance_scores(main['query'], before)
if not before_scores:
    raise RuntimeError('CRAG 首轮检索没有结果')
needs_correction = max(before_scores) < threshold
repair_query = '机器学习算法 绝对 优劣 是否适合 当前待解决的问题'
after = search(repair_query, top_k=4) if needs_correction else before
after_scores = relevance_scores(main['query'], after)
if not after_scores:
    raise RuntimeError('CRAG 重查没有结果')
ordered = sorted(zip(after_scores, after), key=lambda pair: pair[0], reverse=True)
after = [item for _, item in ordered]
after_scores = [score for score, _ in ordered]
main_annotation = load_annotation(main['id'])

print('主要问题：', main['query'])
print('第一次检索（页码，相关分）：', pages_with_scores(before, before_scores))
print('最高分是否低于阈值：', round(max(before_scores), 3), '<', threshold, '→', needs_correction)
print('补查问题：', repair_query if needs_correction else '未触发补查')
print('重新检索并按相关分排序：', pages_with_scores(after, after_scores))
before_found = sorted(set(main_annotation['expected_pages']).intersection(item.page for item in before))
after_found = sorted(set(main_annotation['expected_pages']).intersection(item.page for item in after))
print('必要页：', before_found, '→', after_found)
print('资料量：第一次返回', len(before), '条；补查后返回', len(after), '条；重排检查严格使用本地 snapshot')
assert needs_correction and not before_found and after_found == [17]

check = cases['bellman_value_function']
check_results = search(check['query'], top_k=4)
check_scores = relevance_scores(check['query'], check_results)
if not check_scores:
    raise RuntimeError('CRAG 对照检索没有结果')
check_needs_correction = max(check_scores) < threshold
crag_main_before = before
crag_main_after = after
crag_check_before = check_results
crag_check_after = check_results
print(chr(10) + '复查问题：', check['query'])
print('第一次检索（页码，相关分）：', pages_with_scores(check_results, check_scores))
print('是否需要重新检索：', check_needs_correction)
assert check_results[0].page == 194 and not check_needs_correction
print(chr(10) + '上面的 CRAG 只做本地重排检查；Self-RAG 生成、反思和补查见下方独立实验。')


/usr/local/Caskroom/miniconda/base/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


主要问题： 机器学习算法之间有没有绝对更好的一个？
第一次检索（页码，相关分）： [(15, 0.653), (16, -0.371), (88, -3.861), (18, -4.403)]
最高分是否低于阈值： 0.653 < 1.0 → True
补查问题： 机器学习算法 绝对 优劣 是否适合 当前待解决的问题
重新检索并按相关分排序： [(17, 8.089), (16, -0.371), (18, -4.403), (100, -7.256)]
必要页： [] → [17]
资料量：第一次返回 4 条；补查后返回 4 条；重排检查严格使用本地 snapshot



复查问题： Bellman 等式描述了当前状态价值和未来状态价值的什么关系？
第一次检索（页码，相关分）： [(194, 8.274), (179, -6.095), (58, -7.686), (181, -7.107)]
是否需要重新检索： False

上面的 CRAG 只做本地重排检查；Self-RAG 生成、反思和补查见下方独立实验。


主要问题第一次检索的最高相关分低于阈值，触发固定补查问题后重排；复查问题第一次就超过阈值，因此没有多做一次检索。阈值和补查问题只在这两个案例上观察，不能搬到其他资料库。

## Self-RAG：生成中自我反思并决定是否补查

模型先根据首轮片段生成草稿，再返回严格反思对象：sufficient、missing、repair_queries、supporting_quotes 四个字段必须同时存在且类型正确。sufficient=true 时 missing 和 repair_queries 必须为空，且必须给出必要支持；sufficient=false 时 missing 与 repair_queries 都必须是非空、唯一字符串列表。任何格式或语义不满足都会直接失败。

repair_queries 只执行模型反思返回的语句；不会从原问题或 missing 拼接查询。反思失败与模型正常判断资料不足是两种不同状态：前者阻断，后者才进入正常补查流程。

In [2]:
from common.eval_utils import emit_tutorial_audit, normalize_text
import json
import re

from common.nontraining_utils import (
    build_reused_chunk_search,
    evidence_payload,
    format_context,
    load_annotation,
    load_query_only,
    load_zhipuai_api_key,
    rank_and_coverage,
    unique_evidence,
)

def call_glm_once(prompt: str, *, max_tokens: int = 900, response_format: dict | None = None) -> str:
    '''Make exactly one glm-4-flash request; all failures propagate.'''
    from zhipuai import ZhipuAI

    client = ZhipuAI(api_key=load_zhipuai_api_key(), max_retries=0)
    request = {'model': 'glm-4-flash', 'messages': [{'role': 'user', 'content': prompt}], 'temperature': 0.0, 'max_tokens': max_tokens, 'timeout': 60}
    if response_format is not None:
        request['response_format'] = response_format
    response = client.chat.completions.create(**request)
    choices = getattr(response, 'choices', None)
    if not choices:
        raise RuntimeError('glm-4-flash 没有返回 choices')
    content = getattr(getattr(choices[0], 'message', None), 'content', None)
    if not isinstance(content, str) or not content.strip():
        raise RuntimeError('glm-4-flash 返回空文字')
    return content.strip()

def _strict_json_object(raw: object, stage: str) -> dict:
    if not isinstance(raw, str) or not raw.strip():
        raise ValueError(f'{stage} 必须返回非空 JSON 对象')
    text = raw.strip()
    if text.startswith('```'):
        lines = text.splitlines()
        if len(lines) < 3 or lines[0].strip().lower() not in {'```', '```json'} or lines[-1].strip() != '```':
            raise ValueError(f'{stage} 的 JSON 围栏不完整')
        text = chr(10).join(lines[1:-1]).strip()
    if '```' in text:
        raise ValueError(f'{stage} 包含 JSON 之外的围栏或文字')

    def reject_duplicate_keys(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError(f'{stage} JSON 含重复字段：{key}')
            result[key] = value
        return result

    try:
        value = json.loads(text, object_pairs_hook=reject_duplicate_keys)
    except json.JSONDecodeError as error:
        raise ValueError(f'{stage} 不是合法 JSON：{error}') from error
    except TypeError as error:
        raise ValueError(f'{stage} 不是可解析 JSON：{error}') from error
    if not isinstance(value, dict):
        raise ValueError(f'{stage} 必须是 JSON 对象')
    return value

def _strict_string_list(value: object, field: str, stage: str, *, min_items: int, max_items: int, unique: bool) -> list[str]:
    if not isinstance(value, list) or not min_items <= len(value) <= max_items:
        raise ValueError(f'{stage}.{field} 必须是长度 {min_items}..{max_items} 的字符串列表')
    result = []
    for index, item in enumerate(value):
        if not isinstance(item, str) or not item.strip():
            raise ValueError(f'{stage}.{field}[{index}] 必须是非空字符串')
        result.append(item.strip())
    if unique and len(result) != len(set(result)):
        raise ValueError(f'{stage}.{field} 不得包含重复字符串')
    return result

def parse_self_rag_reflection(raw: str) -> dict:
    stage = 'Self-RAG Reflection'
    payload = _strict_json_object(raw, stage)
    required = {'sufficient', 'missing', 'repair_queries', 'supporting_quotes'}
    if set(payload) != required:
        raise ValueError(f'{stage} 字段必须精确为 sufficient、missing、repair_queries、supporting_quotes')
    sufficient = payload['sufficient']
    if type(sufficient) is not bool:
        raise ValueError(f'{stage}.sufficient 必须是 JSON boolean')
    missing = _strict_string_list(payload['missing'], 'missing', stage, min_items=0, max_items=12, unique=True)
    repair_queries = _strict_string_list(payload['repair_queries'], 'repair_queries', stage, min_items=0, max_items=3, unique=True)
    supporting_quotes = _strict_string_list(payload['supporting_quotes'], 'supporting_quotes', stage, min_items=0, max_items=12, unique=True)
    if sufficient:
        if missing or repair_queries or not supporting_quotes:
            raise ValueError(f'{stage} 足够时 missing 和 repair_queries 必须为空且 supporting_quotes 必须非空')
    elif not missing or not repair_queries:
        raise ValueError(f'{stage} 不足时 missing 和 repair_queries 必须都是非空列表')
    return {'sufficient': sufficient, 'missing': missing, 'repair_queries': repair_queries, 'supporting_quotes': supporting_quotes}

def bind_supporting_quotes(reflection: dict, candidates: list[dict]) -> list[dict]:
    """把反思引文绑定到本轮实际提交给模型的候选文字。"""
    if not isinstance(reflection, dict) or not isinstance(candidates, list) or not candidates:
        raise ValueError('Self-RAG supporting_quotes 绑定需要非空候选资料')
    candidate_texts = []
    for index, candidate in enumerate(candidates):
        if isinstance(candidate, dict):
            text = candidate.get('text')
            page = candidate.get('page')
            chunk_id = candidate.get('chunk_id')
        else:
            text = getattr(candidate, 'text', None)
            page = getattr(candidate, 'page', None)
            chunk_id = getattr(candidate, 'chunk_id', None)
        normalized_candidate = normalize_text(text)
        if not normalized_candidate:
            raise ValueError(f'Self-RAG 候选资料 {index} 为空，不能绑定 supporting_quotes')
        candidate_texts.append({'index': index, 'page': page, 'chunk_id': chunk_id, 'text': normalized_candidate})
    quotes = reflection.get('supporting_quotes')
    if not isinstance(quotes, list):
        raise ValueError('Self-RAG supporting_quotes 必须先通过严格解析器')
    bindings = []
    seen_normalized = set()
    for index, quote in enumerate(quotes):
        normalized_quote = normalize_text(quote)
        if not normalized_quote:
            raise ValueError(f'Self-RAG supporting_quotes[{index}] 归一化后为空')
        if normalized_quote in seen_normalized:
            raise ValueError(f'Self-RAG supporting_quotes[{index}] 与前一条引文归一化后重复')
        seen_normalized.add(normalized_quote)
        matches = [candidate for candidate in candidate_texts if normalized_quote in candidate['text']]
        if not matches:
            raise ValueError(f'Self-RAG supporting_quotes[{index}] 不是本轮候选资料中的逐字引文')
        candidate = matches[0]
        bindings.append({
            'quote': quote,
            'quote_normalized': normalized_quote,
            'candidate_index': candidate['index'],
            'page': candidate['page'],
            **({'chunk_id': candidate['chunk_id']} if candidate['chunk_id'] else {}),
        })
    return bindings


def _must_reject(function, *args):
    try:
        function(*args)
    except (TypeError, ValueError):
        return
    raise AssertionError(f'{function.__name__} 错误输入未被拒绝')

_must_reject(parse_self_rag_reflection, '{}')
_must_reject(parse_self_rag_reflection, json.dumps({'sufficient': 'true', 'missing': [], 'repair_queries': [], 'supporting_quotes': ['q']}))
_must_reject(parse_self_rag_reflection, json.dumps({'sufficient': True, 'missing': [], 'repair_queries': ['q'], 'supporting_quotes': ['q']}))
_must_reject(parse_self_rag_reflection, json.dumps({'sufficient': False, 'missing': [], 'repair_queries': [], 'supporting_quotes': []}))
_must_reject(parse_self_rag_reflection, json.dumps({'sufficient': False, 'missing': ['m', 'm'], 'repair_queries': ['q'], 'supporting_quotes': []}))

CASE_IDS = ['self_rag_cost_threshold', 'self_rag_generalization']
queries = load_query_only(CASE_IDS)
search = build_reused_chunk_search()

well_shaped_fabricated = parse_self_rag_reflection(json.dumps({
    'sufficient': True,
    'missing': [],
    'repair_queries': [],
    'supporting_quotes': ['这是一个形式完整但不在候选资料中的支持引文。'],
}, ensure_ascii=False))
_must_reject(bind_supporting_quotes, well_shaped_fabricated, [{'page': 1, 'text': '候选资料中实际存在的完整支持句。'}])

adversarial_candidates = [
    {'page': 1, 'chunk_id': 'candidate-1', 'text': '候选资料第一段\t只陈述甲。'},
    {'page': 2, 'chunk_id': 'candidate-2', 'text': '候选资料第二段只陈述乙。'},
]
for fabricated_quote in (
    '候选资料第一段只陈述甲。候选资料第二段只陈述乙。',  # 跨候选拼接
    '用户问题中的查询条件并非资料结论。',  # query-only
    '草稿补写的未被资料证明的结论。',  # draft-only
):
    adversarial_reflection = parse_self_rag_reflection(json.dumps({
        'sufficient': True,
        'missing': [],
        'repair_queries': [],
        'supporting_quotes': [fabricated_quote],
    }, ensure_ascii=False))
    _must_reject(bind_supporting_quotes, adversarial_reflection, adversarial_candidates)

normalized_binding = bind_supporting_quotes(parse_self_rag_reflection(json.dumps({
    'sufficient': True,
    'missing': [],
    'repair_queries': [],
    'supporting_quotes': ['候选资料第一段 只陈述甲。'],
}, ensure_ascii=False)), adversarial_candidates)
assert normalized_binding[0]['candidate_index'] == 0 and normalized_binding[0]['quote_normalized'] in adversarial_candidates[0]['text'].replace('\t', ' ')

def cutoff_notice(question, hits):
    asks_for_steps = re.search(r'怎样|如何|步骤|过程', question)
    has_cut_off_text = any(hit.text.strip() and hit.text.strip()[-1] not in '。！？；：）】”' for hit in hits)
    if asks_for_steps and has_cut_off_text:
        return '资料在关键步骤中途截断。只说明资料不足和截断位置，不补写后续内容。'
    return ''

def one_pass(question: str):
    hits = search(question, top_k=1)
    answer = call_glm_once(cutoff_notice(question, hits) + '仅根据下面已经给出的资料回答问题。资料被截断、步骤不完整或没有直接依据时，明确说资料不足，不用常识补写。只回答问题明确询问的内容。' + chr(10) + '问题：' + question + chr(10) + '资料：' + format_context(hits))
    return hits, answer

def self_rag_pipeline(question: str):
    initial_hits = search(question, top_k=1)
    initial_context = format_context(initial_hits, max_chars=3600)
    draft = call_glm_once(cutoff_notice(question, initial_hits) + '仅根据下面已经给出的资料写一版草稿。资料被截断、步骤不完整或没有直接依据时，明确指出缺少什么，不用常识补写。只回答问题明确询问的内容。' + chr(10) + '问题：' + question + chr(10) + '资料：' + initial_context)
    reflection_schema = {'sufficient': True, 'missing': [], 'repair_queries': [], 'supporting_quotes': ['候选资料中的必要原文短语']}
    reflection_true_example = {'sufficient': True, 'missing': [], 'repair_queries': [], 'supporting_quotes': ['直接支持全部要求的原文短语']}
    reflection_false_example = {'sufficient': False, 'missing': ['缺少的独立要求'], 'repair_queries': ['保留专有名词的补查语句'], 'supporting_quotes': []}
    reflection_raw = call_glm_once(
        '你是 Self-RAG 的严格证据反思器。逐项核对用户问题和候选资料。只有每个独立要求都有直接资料支持时 sufficient 才为 true；不能凭主题相近、常识或草稿推断。sufficient 必须是 JSON boolean。sufficient=true 时 missing 和 repair_queries 必须为空且 supporting_quotes 非空；sufficient=false 时 missing 和 repair_queries 都必须是非空、唯一字符串列表，repair_queries 最多 3 条。对象字段必须精确为 sufficient、missing、repair_queries、supporting_quotes，列表元素必须都是非空字符串。以下两个对象只是满足形状的最小合法结构示例，不能代替你依据证据作判断：TRUE 示例=' + json.dumps(reflection_true_example, ensure_ascii=False) + '；FALSE 示例=' + json.dumps(reflection_false_example, ensure_ascii=False) + '。只输出一个合法 JSON 对象，不要 Markdown 围栏、解释或其它字段。' + chr(10) + '精确 schema=' + json.dumps(reflection_schema, ensure_ascii=False) + chr(10) + '用户问题：' + question + chr(10) + '候选资料：' + json.dumps(evidence_payload(initial_hits), ensure_ascii=False) + chr(10) + '当前草稿：' + draft,
        max_tokens=260,
        response_format={'type': 'json_object'},
    )
    reflection = parse_self_rag_reflection(reflection_raw)
    # 先把每一条引文绑定到实际提交的候选 evidence，再允许 sufficient 分支继续。
    supporting_quote_bindings = bind_supporting_quotes(reflection, evidence_payload(initial_hits))
    quote_is_cut_off = any(quote[-1] not in '。！？；：）】”' for quote in reflection['supporting_quotes'])
    if reflection['sufficient'] and quote_is_cut_off and re.search(r'怎样|如何|步骤|过程', question):
        raise ValueError('Self-RAG Reflection sufficient=true 但步骤支持引文被截断')
    trace = [
        {'step': 'retrieve', 'round': 1, 'pages': [hit.page for hit in initial_hits]},
        {'step': 'generate', 'round': 1, 'raw': draft, 'parsed': draft},
        {'step': 'reflect', 'round': 1, 'raw': reflection_raw, 'parsed': reflection, 'supporting_quote_bindings': supporting_quote_bindings},
    ]
    stage_call_counts = {'draft': 1, 'reflection': 1, 'final_answer': 0}
    if reflection['sufficient']:
        final_answer = draft
        final_hits = initial_hits
        stop_reason = 'self_reflection_sufficient'
    else:
        follow_hits = []
        for repair_query in reflection['repair_queries']:
            query_hits = search(repair_query, top_k=4)
            follow_hits.extend(query_hits)
            trace.append({'step': 'retrieve', 'round': 2, 'query': repair_query, 'pages': [hit.page for hit in query_hits]})
        final_hits = unique_evidence(initial_hits + follow_hits, limit=6)
        final_context = format_context(final_hits, max_chars=6000)
        final_answer = call_glm_once('仅根据下面的资料回答问题。只回答问题明确询问的内容；资料没有支持的内容就明确说资料不足，不能补充外部知识。' + chr(10) + '问题：' + question + chr(10) + '资料：' + final_context)
        stage_call_counts['final_answer'] = 1
        trace.append({'step': 'generate', 'round': 2, 'raw': final_answer, 'parsed': final_answer})
        stop_reason = 'self_reflection_requested_retrieval'
    trace.append({'step': 'stop', 'reason': stop_reason, 'pages': [hit.page for hit in final_hits]})
    return initial_hits, final_hits, {
        'draft': {'raw': draft, 'parsed': draft},
        'reflection': {'raw': reflection_raw, 'parsed': reflection},
        'supporting_quote_bindings': supporting_quote_bindings,
        'final_answer': {'raw': final_answer, 'parsed': final_answer},
        'trace': trace,
        'stage_call_counts': stage_call_counts,
    }

records = []
for item in queries:
    before, baseline_answer = one_pass(item['query'])
    _, after, outputs = self_rag_pipeline(item['query'])
    outputs['baseline_answer'] = {'raw': baseline_answer, 'parsed': baseline_answer}
    records.append({'case_id': item['id'], 'query': item['query'], 'before': before, 'after': after, 'model_outputs': outputs})

for record in records:
    record['annotation'] = load_annotation(record['case_id'])
    outputs = record['model_outputs']
    baseline_model_calls = 1
    self_rag_model_calls = sum(outputs['stage_call_counts'].values())
    total_model_calls = baseline_model_calls + self_rag_model_calls
    before = rank_and_coverage(record['before'], record['annotation']['expected_pages'])
    after = rank_and_coverage(record['after'], record['annotation']['expected_pages'])
    reflection = outputs['reflection']['parsed']
    actual_repair_queries = [step['query'] for step in outputs['trace'] if step['step'] == 'retrieve' and step.get('round') == 2]
    report = {
        'case_id': record['case_id'],
        'method': '生成后检查缺口并补查（Self-RAG）',
        'role': 'main' if record['case_id'] == CASE_IDS[0] else 'check',
        'before': before,
        'after': after,
        'model_outputs': outputs,
        'execution_trace': outputs['trace'],
        'stage_call_counts': {'baseline': baseline_model_calls, 'self_rag': outputs['stage_call_counts']},
        'baseline_model_call_count': baseline_model_calls,
        'self_rag_model_call_count': self_rag_model_calls,
        'model_call_count': total_model_calls,
    }
    print(chr(10) + '--- Self-RAG 对照：' + record['query'] + '（模型=glm-4-flash）---')
    emit_tutorial_audit(report)
    print('问题：', record['query'])
    print('流程摘要（根据实际执行记录）：', '首轮资料已覆盖问题，无需补查' if reflection['sufficient'] else '首轮资料不足，按反思契约执行补查；缺少：' + '、'.join(reflection['missing']))
    print('实际执行的检索问题：首轮：', record['query'])
    print('实际执行的补查问题：', '；'.join(actual_repair_queries) if actual_repair_queries else '无')
    print('改动前资料：页', report['before']['pages'], '，', len(record['before']), '个片段')
    print('模型原始回答（基线，截取）：', outputs['baseline_answer']['raw'][:180])
    print('改动后资料：页', report['after']['pages'], '，', len(record['after']), '个片段')
    print('模型原始回答（Self-RAG 草稿，截取）：', outputs['draft']['raw'][:180])
    print('模型原始回答（Self-RAG 最终，截取）：', outputs['final_answer']['raw'][:180])
    print('模型调用次数：基线', baseline_model_calls, '次；Self-RAG', self_rag_model_calls, '次；合计', total_model_calls, '次。')


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



--- Self-RAG 对照：在由 ROC 点转换出的代价曲线图上，怎样根据 P(+)cost 和 costnorm 选择最佳阈值？（模型=glm-4-flash）---


问题： 在由 ROC 点转换出的代价曲线图上，怎样根据 P(+)cost 和 costnorm 选择最佳阈值？
流程摘要（根据实际执行记录）： 首轮资料不足，按反思契约执行补查；缺少：如何根据P(+)cost和costnorm选择最佳阈值的详细步骤和计算方法
实际执行的检索问题：首轮： 在由 ROC 点转换出的代价曲线图上，怎样根据 P(+)cost 和 costnorm 选择最佳阈值？
实际执行的补查问题： 请提供关于如何根据P(+)cost和costnorm选择最佳阈值的详细步骤和计算方法的解释
改动前资料：页 [25] ， 1 个片段
模型原始回答（基线，截取）： 资料不足，无法回答问题。
改动后资料：页 [25, 25, 26, 24, 24] ， 5 个片段
模型原始回答（Self-RAG 草稿，截取）： 资料不足：缺少关于如何根据P(+)cost和costnorm选择最佳阈值的详细步骤和计算方法。截断位置：在“再基于该点作一条垂”之后。
模型原始回答（Self-RAG 最终，截取）： 在由ROC点转换出的代价曲线图上，选择最佳阈值的方法如下：

1. 首先计算当前样例集的P(+)cost值。
2. 在横轴上标记出具体的点。
3. 基于该点作一条垂直于横轴的垂线。
4. 与该垂线最先相交（从下往上看）的线段所对应的阈值即为最佳阈值。
5. 原因是与该垂线最先相交的线段必然最靠下，因此其交点的纵坐标最小，而纵轴表示的便是归一化代价costno
模型调用次数：基线 1 次；Self-RAG 3 次；合计 4 次。

--- Self-RAG 对照：训练集上的经验误差和新样本上的泛化误差分别表示什么？（模型=glm-4-flash）---


问题： 训练集上的经验误差和新样本上的泛化误差分别表示什么？
流程摘要（根据实际执行记录）： 首轮资料已覆盖问题，无需补查
实际执行的检索问题：首轮： 训练集上的经验误差和新样本上的泛化误差分别表示什么？
实际执行的补查问题： 无
改动前资料：页 [18] ， 1 个片段
模型原始回答（基线，截取）： 训练集上的经验误差表示学习器在训练集上预测错误的比例，即训练集上差异的平均值。新样本上的泛化误差表示学习器在新样本（训练集中未出现过的样本）上预测错误的比例，即在新样本上差异的平均值。
改动后资料：页 [18] ， 1 个片段
模型原始回答（Self-RAG 草稿，截取）： 训练集上的经验误差表示学习器在训练集上预测结果与实际结果不一致的情况的平均程度。具体来说，它是通过计算训练集中所有样本的实际类别与学习器预测的类别之间的差异（不一致为1，一致为0），然后取这些差异的平均值来定义的。

新样本上的泛化误差表示学习器在新样本（即训练集中未出现过的样本）上预测结果与实际结果不一致的情况的平均程度。与经验误差类似，泛化误差也是通过计
模型原始回答（Self-RAG 最终，截取）： 训练集上的经验误差表示学习器在训练集上预测结果与实际结果不一致的情况的平均程度。具体来说，它是通过计算训练集中所有样本的实际类别与学习器预测的类别之间的差异（不一致为1，一致为0），然后取这些差异的平均值来定义的。

新样本上的泛化误差表示学习器在新样本（即训练集中未出现过的样本）上预测结果与实际结果不一致的情况的平均程度。与经验误差类似，泛化误差也是通过计
模型调用次数：基线 1 次；Self-RAG 2 次；合计 3 次。


## CRAG 和 Self-RAG 的区别

CRAG 在生成前用本地 BGE 重排分数检查首轮资料；Self-RAG 启发式流程在生成草稿后检查问题覆盖、资料支持和缺口。CRAG 的重排输出与 Self-RAG 的 `glm-4-flash` 输出是两种独立实验，不能互相替代。

## 运行边界

CRAG 运行只接受本地 reranker snapshot，缺少文件立即抛出 FileNotFoundError；下载资源只能在运行前按 README 命令准备。Self-RAG 的每次模型请求都固定为 `glm-4-flash` 的单次调用，严格响应解析失败立即停止；只有模型明确返回 sufficient=false 且同时给出合法缺口与补查语句时，才执行补查。

In [3]:
from common.eval_utils import emit_tutorial_audit

def _actual_pages(items):
    pages = []
    for item in items:
        values = item.pages if hasattr(item, 'pages') else [item.page]
        for page in values:
            page = int(page)
            if page not in pages:
                pages.append(page)
    return pages

def _metrics(items, expected_pages):
    pages = _actual_pages(items)
    expected = {int(page) for page in expected_pages}
    found = set(pages) & expected
    rank = next((index for index, page in enumerate(pages, 1) if page in expected), None)
    return {'pages': pages, 'first_required_rank': rank, 'required_page_coverage': len(found) / len(expected) if expected else 0.0}

def _emit_crag(role, case_id, before_items, after_items, purpose=None):
    annotation = load_annotation(case_id)
    payload = {'case_id': case_id, 'method': '检查相关性后重新检索（CRAG）', 'role': role, 'before': _metrics(before_items, annotation['expected_pages']), 'after': _metrics(after_items, annotation['expected_pages']), 'reranker': {'model_id': RERANKER_MODEL_ID, 'snapshot': RERANKER_MODEL_ID, 'local_files_only': True}}
    if purpose:
        payload['check_purpose'] = purpose
    emit_tutorial_audit(payload)

_emit_crag('main', 'model_selection_no_absolute_best', crag_main_before, crag_main_after)
_emit_crag('check', 'bellman_value_function', crag_check_before, crag_check_after, '确认没有改坏')


## Valid insufficient：资料缺失时只拒答

下面的问题明确要求资料没有提供的 CUDA 版本。回答模型仍然只接收真实检索上下文，并通过项目根 `.env` 的 `ZHIPUAI_API_KEY` 调用字面量 `glm-4-flash`；不提供备用模型、默认答案或外部知识。保存 `raw_response`、解析对象和审计 MIME，只有解析为 `status=insufficient` 且 `claims=[]` 才算通过。原有 Self-RAG reflection 的 `sufficient`、`missing`、`repair_queries`、`supporting_quotes` 四字段契约及 supporting quotes 与真实候选原文的绑定保持不变。


In [4]:
missing_question = '《南瓜书》建议使用哪个 CUDA 版本训练模型？'
missing_hits = search(missing_question, top_k=1)
if not missing_hits:
    raise RuntimeError('资料缺失用例没有返回检索结果')
missing_schema = {'status': 'answerable|conflict|insufficient', 'answer': '非空中文回答', 'claims': ['只有可回答时才填写的逐字证据']}
missing_prompt = (
    '你是严格的资料受限回答器。只能根据 QUESTION 和 CONTEXT 回答，不能使用外部知识。'
    '输出必须是一个 JSON 对象，字段必须恰好且完整地只有 status、answer、claims 三个，任何字段都不得省略，也不得增加字段。'
    'status 只能是 answerable、conflict 或 insufficient；answer 必须是非空字符串；claims 必须是 JSON 数组。'
    '若 CONTEXT 没有直接回答 QUESTION，status 必须为 insufficient，claims 必须为空数组，answer 必须明确说明资料不足且不得猜测。'
    '以下仅为字段和类型约束，不是答案：' + json.dumps(missing_schema, ensure_ascii=False)
    + chr(10) + 'QUESTION：' + missing_question
    + chr(10) + 'CONTEXT：' + format_context(missing_hits)
    + chr(10) + '现在仅输出满足上述精确三字段契约的 JSON 对象。'
)
missing_raw_response = call_glm_once(missing_prompt, max_tokens=300, response_format={'type': 'json_object'})
missing_parsed_response = _strict_json_object(missing_raw_response, 'Valid insufficient')
required_missing_fields = {'status', 'answer', 'claims'}
if set(missing_parsed_response) != required_missing_fields:
    raise ValueError('Valid insufficient 响应字段必须精确为 status、answer、claims')
if missing_parsed_response['status'] != 'insufficient':
    raise ValueError(f'缺失资料问题必须真实解析为 status=insufficient：{missing_parsed_response!r}')
if missing_parsed_response['claims'] != []:
    raise ValueError(f'insufficient 必须保持 claims=[]，拒绝伪造证据：{missing_parsed_response!r}')
if not isinstance(missing_parsed_response['answer'], str) or not any(word in missing_parsed_response['answer'] for word in ('资料不足', '无法确定', '没有说明')):
    raise ValueError('insufficient answer 必须明确拒答，不得夹带猜测')
missing_audit = {
    'case_id': 'c6_valid_insufficient_cuda_version',
    'method': '资料缺失时严格拒答',
    'model': 'glm-4-flash',
    'api_key_source': '.env:ZHIPUAI_API_KEY',
    'question': missing_question,
    'retrieved': [{'page': hit.page, 'chunk_id': getattr(hit, 'chunk_id', None), 'text': hit.text} for hit in missing_hits],
    'raw_response': missing_raw_response,
    'parsed': missing_parsed_response,
    'status': missing_parsed_response['status'],
    'claims': missing_parsed_response['claims'],
    'no_fabricated_evidence': True,
}
emit_tutorial_audit(missing_audit)
print('Valid insufficient 真实调用已保存：raw_response、parsed、audit；status=', missing_audit['status'], '；claims=', missing_audit['claims'])


Valid insufficient 真实调用已保存：raw_response、parsed、audit；status= insufficient ；claims= []
